# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/nikos/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/nikos/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"
print(os.environ["LANGCHAIN_PROJECT"])

AIM - SDG - 1719080e


OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data/MentalHealthGuide.txt', 'data/HealthWellnessGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/nikos/n/rvm/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/nikos/n/rvm/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/nikos/n/rvm/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 9, relationships: 17)

We can save and load our knowledge graphs as follows.

In [11]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 9, relationships: 17)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
**SingleHopSpecific:** It uses single relationships from the KG and asks direct questions about them e.g., (exercise, --helps with-->, weight loss) could generate "What does exercise help with?" or "what helps with weight loss?"

**MultiHopAbstract:** It asks conceptual questions by looking at depth and breadth that would require the LLM to make the connection across edges, e.g., (exercise --helps with-->, weight loss) (bench press -- is --> exercise), (gym, --has--> bench press) generate "how can I lose weight?"

**MultiHopSpecific:** It links together multiple edges of the KG to ask specific questions about facts by following the links directly. From the example above: "where can I exercise to lose weight?"

Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"As a wellness coach, how does mental health in...",[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
1,What is the correct spelling of DBT?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,The context does not provide information about...,single_hop_specifc_query_synthesizer
2,What role does Vitamin D play in mental health...,[Write letters to or from your future self Jou...,"Vitamin D, obtained from sunlight and fortifie...",single_hop_specifc_query_synthesizer
3,What is the correct spelling of the term APPEN...,[social interactions How to set and maintain b...,"The term is spelled APPENDIX, as indicated in ...",single_hop_specifc_query_synthesizer
4,What does Chapter 9 cover in the context of we...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,The context provided does not include informat...,single_hop_specifc_query_synthesizer
5,how journaling practices and self reflection h...,[<1-hop>\n\nWrite letters to or from your futu...,"Journaling practices, like writing honestly an...",multi_hop_abstract_query_synthesizer
6,How does sleep hygiene influence sleep quality...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Sleep hygiene involves habits and practices th...,multi_hop_abstract_query_synthesizer
7,How can I plan my meals and build a workout ro...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,"To enhance your overall wellness, start by imp...",multi_hop_abstract_query_synthesizer
8,How can understanding the science of habit for...,[<1-hop>\n\nsocial interactions How to set and...,Understanding the science of habit formation h...,multi_hop_specific_query_synthesizer
9,"How does vitamin D, as a key nutrient discusse...",[<1-hop>\n\nWrite letters to or from your futu...,Vitamin D is highlighted as an essential nutri...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What are the recommended exercises and strateg...,[The Personal Wellness Guide A Comprehensive R...,The provided context does not include specific...,single_hop_specifc_query_synthesizer
1,What does Stage 2 of sleep involve in the slee...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Stage 2 involves a drop in body temperature an...,single_hop_specifc_query_synthesizer
2,What information does Chapter 18 cover regardi...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 18 discusses strategies to boost immun...,single_hop_specifc_query_synthesizer
3,How does the World Health Organization define ...,[The Mental Health and Psychology Handbook A P...,"According to the World Health Organization, me...",single_hop_specifc_query_synthesizer
4,how can exercise for common problems like lowe...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,The wellness guide explains that gentle exerci...,multi_hop_abstract_query_synthesizer
5,How can incorporating mindfulness and social c...,[<1-hop>\n\nhour before bed - No caffeine afte...,Incorporating mindfulness and social connectio...,multi_hop_abstract_query_synthesizer
6,How can improving face-to-face interactions an...,[<1-hop>\n\nhour before bed - No caffeine afte...,Improving face-to-face interactions by engagin...,multi_hop_abstract_query_synthesizer
7,How can I improve my emotional intelligence an...,[<1-hop>\n\nhour before bed - No caffeine afte...,To improve emotional intelligence and manage c...,multi_hop_abstract_query_synthesizer
8,how chapter 7 and 17 connect about sleep and h...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,"chapter 7 talks about sleep and recovery, expl...",multi_hop_specific_query_synthesizer
9,H0w c4n I bUild a he4lthy m0rn1ng r0utine (cha...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,To build a healthy morning routine that improv...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
In the manual approach I have more control on the types of queries I want to generate since I can a) select the query synthesizer of my choice and b) decide the percentage distribution of the questions. If I were testing my agent for the first time I'd do the abstracted version to get a baseline evaluation. Then I would look at any weak areas (say multi hop abstract) and then use the unrolled version to dig into this more by generating those types of test cases specifically.

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
custom_query_distribution = [
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 1.0)
]
# Generate a new test set and compare with the default
custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)
custom_testset.to_pandas()
# Observation: All 10 questions are more complex and high-level as expected.

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How can Cognitive Behavioral Therapy (CBT) tec...,[<1-hop>\n\nThe Mental Health and Psychology H...,"Cognitive Behavioral Therapy (CBT) techniques,...",multi_hop_abstract_query_synthesizer
1,How can building healthy habits through consis...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Building healthy habits through consistent rou...,multi_hop_abstract_query_synthesizer
2,How does exercise influence mental health thro...,[<1-hop>\n\nThe Mental Health and Psychology H...,Exercise improves mental health by increasing ...,multi_hop_abstract_query_synthesizer
3,how do build meaningful relashionships thru vu...,[<1-hop>\n\nWrite letters to or from your futu...,The context emphasizes the importance of build...,multi_hop_abstract_query_synthesizer
4,how to recognize weak boundaries and manage di...,[<1-hop>\n\nWrite letters to or from your futu...,"from the first segment, recognizing weak bound...",multi_hop_abstract_query_synthesizer
5,How can understanding habit formation and buil...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Understanding habit formation and building hea...,multi_hop_abstract_query_synthesizer
6,how mental health and well-being connect with ...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that mental health includ...,multi_hop_abstract_query_synthesizer
7,How does sleep influence mental health and rec...,[<1-hop>\n\nWrite letters to or from your futu...,Sleep has a significant impact on mental healt...,multi_hop_abstract_query_synthesizer
8,"How does sleep influence mental health, and wh...",[<1-hop>\n\nWrite letters to or from your futu...,Sleep and mental health have a bidirectional r...,multi_hop_abstract_query_synthesizer
9,how build healthy habits and make morning rout...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,building healthy habits and good morning routi...,multi_hop_abstract_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [18]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"
print(dataset_name)
langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

Use Case Synthetic Data - AIE9 - 64c3f3cd-bf06-4586-b002-578c84ad6a77


We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [19]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [20]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [23]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [25]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [28]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [29]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [30]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`: is the answer factually correct?
> - `labeled_helpfulness_evaluator`: is the information returned relevant and actionable wrt to the question?
> - `dopeness_evaluator`: is the answer's tone engaging?

## LangSmith Evaluation

In [31]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'frosty-backpack-42' at:
https://smith.langchain.com/o/8a0b2b44-c37f-4424-8c22-6130df0e91ad/datasets/23471531-70cd-4a39-a72d-4297b52a184f/compare?selectedSessions=73557a97-06bc-4554-9413-940d71e974d2




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can understanding Chapter 16's digital men...,Understanding Chapter 16's digital mental heal...,None,Understanding Chapter 16's strategies for mana...,True,True,False,3.961940,c9e483c6-c71f-49ce-8c03-fc5ec62bde80,019c5b65-9b4f-7930-adac-1742a60538cf
1,H0w c4n a m3ntal health adv0c4t u5e ch4pt3rs 1...,Based on the context:\n\nA mental health advoc...,None,A m3nt4l h34lth adv0c4t c4n us3 ch4pt3rs 16 4n...,False,True,True,5.280333,1362f331-e8b4-4ae2-847f-75041950c651,019c5b65-e0b8-7180-a6c9-676e332a7867
2,how to set boundaries and manage digital menta...,"Based on the provided context, here is how to ...",None,"to set boundaries, identify your limits, commu...",True,True,False,5.200406,0719ecb8-65f5-45b0-a8c5-1c3a0fd881a6,019c5b66-3292-7e50-ad41-51bb69133bb8
3,How can understanding the effects of social me...,I don't know.,None,Chapter 16 discusses the impact of social medi...,False,False,False,1.358899,d38c33ff-ce47-4d3f-8699-98fbb1d24a9e,019c5b66-80a2-70c3-88c5-99116c739867
4,what types of exercise help with mental health...,The types of exercise that help with mental he...,None,The Personal Wellness Guide states that the fo...,True,True,False,2.132863,bf6be46a-fdb4-4bf8-8991-7a2c48c38eb4,019c5b66-a773-7a33-b5e1-4b50460e3bc5
5,How can understanding the symptoms of mental h...,Understanding the symptoms of mental health co...,None,Understanding the symptoms of mental health co...,True,True,True,2.864718,b82bcb98-381e-4bda-8ef9-6330542acb25,019c5b66-d805-7b41-80d9-dc68425e67b0
6,"How can mindfulness-based therapies, such as M...","Mindfulness-based therapies, including Mindful...",None,Mindfulness-based therapies like MBSR have bee...,True,True,True,1.632836,6f244882-45ea-4126-83b8-3750586af94b,019c5b67-0fda-7fd3-8642-d572f37e6e8b
7,How can recognizing signs of weak boundaries a...,Recognizing signs of weak boundaries and manag...,None,Recognizing signs of weak boundaries—such as f...,True,True,True,3.921676,81dcda43-5e24-4eef-a699-27f77e2b4c9b,019c5b67-40c0-7642-9a22-fb3bcf886cf1
8,What information is included in the appendix r...,The appendix regarding mental health resources...,None,The appendix includes mental health resources ...,False,False,False,3.326666,675cedca-0b7a-44b6-8be0-ea1a69808392,019c5b67-8769-7a01-9372-b1fe1cf91c7c
9,"How does sleep influence mental health, especi...",Sleep influences mental health in several impo...,None,Sleep and mental health have a bidirectional r...,True,True,False,2.324411,e1a0903b-c2c4-46a6-9876-08f9a4d33cdb,019c5b67-b80c-7cb1-9021-66aff10bce87


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [32]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [33]:
rag_documents = docs

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
If we increase our chunk size then we will provide the LLM with more context when they fetch a chunk and as a result they could be making multi-hop connections easier. If we decrease the chunk size we may be able to get more accurate fact based answers potentially since our embeddings for those chunks will have less noise.

In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
A larger embedding model has more dimensions which means it has been trained to capture more relationships and finer-grained details about a text. It will be able to disambiguate between concepts which can be useful for the more abstract questions that we ask our RAG. Even the multihop connections. Which means we can capture more meaning from larger chunks (less "blurring" of the features). Of course, the larger the model the larger the cost of running it and slower performance during inference.

In [36]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

"Alright, let’s crank your sleep game up to legendary status! Based on the wisdom from the sleep sages in Chapter 8 of your HealthWellnessGuide, here’s the blueprint to level up your nightly Zzz’s:\n\n1. **Lock in your sleep schedule** like it’s your squad’s secret meeting—same bedtime and wake-up every day, weekends included. Your body LOVES consistency.\n\n2. **Establish a chill bedtime routine** — think low-key vibes like reading a book, gentle stretching, or soaking in a warm bath. This signals your brain it’s downtime, no exceptions.\n\n3. **Make your bedroom a sleep fortress**: keep it cool (65-68°F/18-20°C), pitch-black with blackout curtains or a slick sleep mask, and absolutely no noise intruders—white noise machines or earplugs have your back.\n\n4. **Ditch screens 1-2 hours before bed**—blue light? Nightmare fuel. Swap the scroll for something zen-focused.\n\n5. **Cut the caffeine after 2 PM**—your coffee’s a daytime hero, but it’s chaos for your nighttime rest.\n\n6. **Exer

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'only-stop-32' at:
https://smith.langchain.com/o/8a0b2b44-c37f-4424-8c22-6130df0e91ad/datasets/23471531-70cd-4a39-a72d-4297b52a184f/compare?selectedSessions=1a4755c5-b4ff-421d-97de-782e5b6daa68




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can understanding Chapter 16's digital men...,"Alright, buckle up — let’s make this mental he...",None,Understanding Chapter 16's strategies for mana...,True,True,True,4.362742,c9e483c6-c71f-49ce-8c03-fc5ec62bde80,019c5ba7-ea0e-7552-95f7-efded089a2f8
1,H0w c4n a m3ntal health adv0c4t u5e ch4pt3rs 1...,"Yo, a mental health advocate rocking chapters ...",None,A m3nt4l h34lth adv0c4t c4n us3 ch4pt3rs 16 4n...,False,False,True,4.462451,1362f331-e8b4-4ae2-847f-75041950c651,019c5ba8-260f-74e1-93a1-7424b5d141fd
2,how to set boundaries and manage digital menta...,"Yo, here’s your ultimate boundary and digital ...",None,"to set boundaries, identify your limits, commu...",True,True,True,5.565514,0719ecb8-65f5-45b0-a8c5-1c3a0fd881a6,019c5ba8-65ff-7473-81ad-bd447f01f089
3,How can understanding the effects of social me...,"Yo, let’s crank this up to eleven with some ne...",None,Chapter 16 discusses the impact of social medi...,True,True,True,4.908037,d38c33ff-ce47-4d3f-8699-98fbb1d24a9e,019c5ba8-a1c0-7cb2-9cc2-41172ad2f2d3
4,what types of exercise help with mental health...,"Alright, here’s the scoop—exercise isn’t just ...",None,The Personal Wellness Guide states that the fo...,True,True,True,3.898375,bf6be46a-fdb4-4bf8-8991-7a2c48c38eb4,019c5ba8-f69b-7112-b938-e70b36eb447e
5,How can understanding the symptoms of mental h...,"Alright, buckle up for some next-level mental ...",None,Understanding the symptoms of mental health co...,True,True,True,3.444455,b82bcb98-381e-4bda-8ef9-6330542acb25,019c5ba9-2962-7370-87be-0828ca471980
6,"How can mindfulness-based therapies, such as M...","Alright, strap in—let’s dive deep into the rad...",None,Mindfulness-based therapies like MBSR have bee...,True,True,True,4.609945,6f244882-45ea-4126-83b8-3750586af94b,019c5ba9-57a2-7510-9a07-fff3f88172ca
7,How can recognizing signs of weak boundaries a...,"Yo, here’s the lowdown on leveling up your emo...",None,Recognizing signs of weak boundaries—such as f...,True,True,True,3.928903,81dcda43-5e24-4eef-a699-27f77e2b4c9b,019c5ba9-90e6-7ca3-9280-73991423f746
8,What information is included in the appendix r...,"Alright, buckle up for a mind-blowing dive int...",None,The appendix includes mental health resources ...,False,False,True,3.678202,675cedca-0b7a-44b6-8be0-ea1a69808392,019c5ba9-bc4e-7bc2-ac25-9e6fc65be316
9,"How does sleep influence mental health, especi...","Oh, buckle up for this mind-glowing sleep saga...",None,Sleep and mental health have a bidirectional r...,True,True,True,2.476007,e1a0903b-c2c4-46a6-9876-08f9a4d33cdb,019c5baa-01ab-7350-806d-76cf6d3eb3ed


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
##### Original RAG Chain:
![rag_chain](./rag_chain.png)

##### Dopeness RAG Chain:
![dopeness_rag_chain](./dopeness_rag_chain.png)

##### Comparison
- *Dopeness*: increased to **1.0** from **0.333** (That's expected since we modified the prompt explicitly about that and as we can see all the answers look super dope compared to the no-dope-at-all english of the original)

- *Helpfulness*: increased to **0.833** from **0.75** a 12% improvement (That's expected since both chunk size and embedding model are bigger and it's more likely to dig up connections to useful information. However it's only a 13% improvement even though we have 4x the size (2x chunk size) x (2x embedding model))

- *QA*: increased to **0.75** from **0.667** a 12% improvement (That's expected since we have more accurate embeddings but it's interesting that the return on investment in 2x'ing our embedding model is not that great.)

- *Latency*: increased to **3.91** from **2.60** average which is 50% (the large model has 2x the parameters of the small embedding-3 model so it makes sense but at the same time we also increased our chunk size which means we would be retrieving fewer docs compared to the original RAG so that's why we only see a discount. Even though we 2x the chunk size it doesn't mean we got half the entries. Our splitter is recursive so it's possible we had a tail of smaller chunks during chunking. That analysis assumes all network conditions were the same and there are no other hidden factors during the API calls or if OpenAI uses some smart caching to speed things up)

- *Tokens*: We see a drop in tokens by 12% (This makes sense since we have fewer docs to put in our context to the LLM)

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores